# Read and Process Data

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.master("yarn").appName("CustomerDataProcessing").getOrCreate()
spark

26/06/20 05:14:55 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
!hadoop fs -ls /data


Found 6 items
-rw-r--r--   2 root hadoop    1058478 2026-06-04 17:20 /data/customers.csv
-rw-r--r--   2 root hadoop   26214400 2026-06-04 17:48 /data/customers_500.csv
-rw-r--r--   2 root hadoop        209 2026-06-07 07:42 /data/dates_data.csv
-rw-r--r--   2 root hadoop     653137 2026-06-06 16:16 /data/messy_customer_data.csv
drwxr-xr-x   - root hadoop          0 2026-06-06 16:55 /data/write_output.csv
drwxr-xr-x   - root hadoop          0 2026-06-06 17:00 /data/write_output1.csv


In [4]:
df = spark.read.format('csv').option('header','true').load("/data/customers.csv")

In [5]:
df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-05-17|     True|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-11-29|     True|
|          2|Customer_2|Hyderabad|Maharashtra|  India|       2023-02-08|     True|
|          3|Customer_3|Bangalore|Maharashtra|  India|       2023-03-28|    False|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-06-04|    False|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [6]:
df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- is_active: string (nullable = true)



In [18]:
from pyspark.sql.functions import *

In [8]:
df = df.withColumn('registration_date',to_date(col('registration_date'),'yyyy-MM-dd')) \
    .withColumn('is_active',col('is_active').cast('boolean'))
df.show(5)
df.printSchema()

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-05-17|     true|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-11-29|     true|
|          2|Customer_2|Hyderabad|Maharashtra|  India|       2023-02-08|     true|
|          3|Customer_3|Bangalore|Maharashtra|  India|       2023-03-28|    false|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-06-04|    false|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nu

In [9]:
df = df.fillna({'city':'Unknown','state':'Unknown','country':'Unknown'})

In [10]:
df = df.withColumn('registraion_year',year(col('registration_date'))) \
       .withColumn('registraion_month',month(col('registration_date')))
df.show(5)
       

+-----------+----------+---------+-----------+-------+-----------------+---------+----------------+-----------------+
|customer_id|      name|     city|      state|country|registration_date|is_active|registraion_year|registraion_month|
+-----------+----------+---------+-----------+-------+-----------------+---------+----------------+-----------------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-05-17|     true|            2023|                5|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-11-29|     true|            2023|               11|
|          2|Customer_2|Hyderabad|Maharashtra|  India|       2023-02-08|     true|            2023|                2|
|          3|Customer_3|Bangalore|Maharashtra|  India|       2023-03-28|    false|            2023|                3|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-06-04|    false|            2023|                6|
+-----------+----------+---------+-----------+-------+--

In [11]:
unique_cities = df.select(countDistinct('city')).collect()
unique_cities[0][0]

unique_states = df.select(countDistinct('state')).collect()
unique_states[0][0]

unique_countries = df.select(countDistinct('country')).collect()
unique_countries[0][0]

print(unique_cities[0][0])
print(unique_states[0][0])
print(unique_countries[0][0])

8
7
1


In [12]:
df.groupBy('city').count().orderBy(col('count').desc()).show()

+---------+-----+
|     city|count|
+---------+-----+
|     Pune| 2216|
|Hyderabad| 2194|
|  Kolkata| 2185|
|Bangalore| 2175|
|    Delhi| 2161|
|Ahmedabad| 2157|
|  Chennai| 2151|
|   Mumbai| 2093|
+---------+-----+



In [13]:
df.groupBy('country','state').count().orderBy(col('count')).show()

+-------+-----------+-----+
|country|      state|count|
+-------+-----------+-----+
|  India|West Bengal| 2440|
|  India|Maharashtra| 2449|
|  India|  Karnataka| 2449|
|  India|    Gujarat| 2478|
|  India|  Telangana| 2494|
|  India| Tamil Nadu| 2510|
|  India|      Delhi| 2512|
+-------+-----------+-----+



In [14]:
#Pivot Table - count of active or inactive users per state

df.groupBy('state').pivot('is_active').count().show()

+-----------+-----+----+
|      state|false|true|
+-----------+-----+----+
|    Gujarat| 1198|1280|
|      Delhi| 1271|1241|
|  Karnataka| 1252|1197|
|  Telangana| 1229|1265|
|Maharashtra| 1243|1206|
| Tamil Nadu| 1303|1207|
|West Bengal| 1280|1160|
+-----------+-----+----+



In [19]:
from pyspark.sql.window import Window

In [20]:
window_spec = Window.partitionBy('state').orderBy(col('registration_date').desc())

df= df.withColumn('rank',rank().over(window_spec))\
    .withColumn('dense_rank',dense_rank().over(window_spec))\
        .withColumn('row_number',row_number().over(window_spec))

In [21]:
df.show(5)

+-----------+--------------+---------+-----+-------+-----------------+---------+----------------+-----------------+----+----------+----------+
|customer_id|          name|     city|state|country|registration_date|is_active|registraion_year|registraion_month|rank|dense_rank|row_number|
+-----------+--------------+---------+-----+-------+-----------------+---------+----------------+-----------------+----+----------+----------+
|        567|  Customer_567|  Chennai|Delhi|  India|       2023-12-31|    false|            2023|               12|   1|         1|         1|
|        709|  Customer_709|     Pune|Delhi|  India|       2023-12-31|    false|            2023|               12|   1|         1|         2|
|       5269| Customer_5269|Bangalore|Delhi|  India|       2023-12-31|     true|            2023|               12|   1|         1|         3|
|       6347| Customer_6347|  Kolkata|Delhi|  India|       2023-12-31|     true|            2023|               12|   1|         1|         4|

In [26]:
df_recent_customers = df.filter(col('registration_date') >= lit('2023-12-31'))
df_recent_customers.show(5)

+-----------+--------------+---------+-----+-------+-----------------+---------+----------------+-----------------+----+----------+----------+
|customer_id|          name|     city|state|country|registration_date|is_active|registraion_year|registraion_month|rank|dense_rank|row_number|
+-----------+--------------+---------+-----+-------+-----------------+---------+----------------+-----------------+----+----------+----------+
|        567|  Customer_567|  Chennai|Delhi|  India|       2023-12-31|    false|            2023|               12|   1|         1|         1|
|        709|  Customer_709|     Pune|Delhi|  India|       2023-12-31|    false|            2023|               12|   1|         1|         2|
|       5269| Customer_5269|Bangalore|Delhi|  India|       2023-12-31|     true|            2023|               12|   1|         1|         3|
|       6347| Customer_6347|  Kolkata|Delhi|  India|       2023-12-31|     true|            2023|               12|   1|         1|         4|

In [27]:
# oldest and newest customer per city

In [28]:
df.groupBy('city').agg(min('registration_date').alias('oldest'),max('registration_date').alias('newest')).show()

+---------+----------+----------+
|     city|    oldest|    newest|
+---------+----------+----------+
|    Delhi|2023-01-01|2023-12-31|
|  Kolkata|2023-01-01|2023-12-31|
|Hyderabad|2023-01-01|2023-12-31|
|Bangalore|2023-01-01|2023-12-31|
|Ahmedabad|2023-01-01|2023-12-31|
|  Chennai|2023-01-01|2023-12-31|
|   Mumbai|2023-01-01|2023-12-31|
|     Pune|2023-01-01|2023-12-31|
+---------+----------+----------+

